<a href="https://colab.research.google.com/github/tai201120112010/Train-Model/blob/main/Medical_QA_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Medical_QA_Project


In [ ]:
# =========================================================
# 1. CÀI THƯ VIỆN
# =========================================================

# transformers: chứa các mô hình NLP như BERT
# datasets: dùng để tải dataset NLP
# accelerate: tăng tốc training trên GPU

!pip install -U transformers datasets accelerate


# =========================================================
# 2. IMPORT THƯ VIỆN
# =========================================================

from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForQuestionAnswering
from transformers import TrainingArguments, Trainer
from transformers import pipeline


# =========================================================
# 3. LOAD DATASET
# =========================================================

# SQuAD là dataset chuẩn cho Question Answering
dataset = load_dataset("squad")

print(dataset)


# =========================================================
# 4. LOAD TOKENIZER
# =========================================================

# tokenizer chuyển text -> token (số) để model xử lý

model_checkpoint = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


# =========================================================
# 5. PREPROCESS DATASET (PHẦN QUAN TRỌNG)
# =========================================================

# Bước này sẽ:
# - tokenize question + context
# - tìm vị trí answer trong token
# - tạo start_positions và end_positions

def preprocess_function(examples):

    questions = [q.strip() for q in examples["question"]]

    # tokenize question + context
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=384,
        truncation="only_second",
        padding="max_length",
        return_offsets_mapping=True
    )

    offset_mapping = inputs.pop("offset_mapping")
    answers = examples["answers"]

    start_positions = []
    end_positions = []

    # duyệt từng sample trong batch
    for i, offset in enumerate(offset_mapping):

        answer = answers[i]

        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        sequence_ids = inputs.sequence_ids(i)

        # tìm vị trí bắt đầu của context
        context_start = 0
        while sequence_ids[context_start] != 1:
            context_start += 1

        # tìm vị trí kết thúc context
        context_end = len(sequence_ids) - 1
        while sequence_ids[context_end] != 1:
            context_end -= 1

        token_start = context_start
        token_end = context_end

        # tìm token start của answer
        while token_start <= context_end and offset[token_start][0] <= start_char:
            token_start += 1

        start_positions.append(token_start - 1)

        # tìm token end của answer
        while offset[token_end][1] >= end_char:
            token_end -= 1

        end_positions.append(token_end + 1)

    # thêm label vào dataset
    inputs["start_positions"] = start_positions
    inputs["end_positions"] = end_positions

    return inputs


# áp dụng preprocessing cho dataset
tokenized_datasets = dataset.map(preprocess_function, batched=True)


# =========================================================
# 6. KIỂM TRA DATASET SAU PREPROCESS
# =========================================================

print(tokenized_datasets["train"].column_names)

# phải thấy các cột:
# input_ids
# attention_mask
# start_positions
# end_positions


# =========================================================
# 7. LOAD MODEL BERT QA
# =========================================================

model = AutoModelForQuestionAnswering.from_pretrained(
    "bert-base-uncased"
)


# =========================================================
# 8. CẤU HÌNH TRAINING
# =========================================================

training_args = TrainingArguments(
    output_dir="medical_qa_model",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    num_train_epochs=2,
    logging_steps=100
)


# =========================================================
# 9. TẠO TRAINER
# =========================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"]
)


# =========================================================
# 10. TRAIN MODEL
# =========================================================

trainer.train()




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 110.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

plain_text/validation-00000-of-00001.par(…):   0%|          | 0.00/1.82M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyError: "Unknown task question-answering, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [ ]:
# =========================================================
# 13. LƯU MODEL
# =========================================================

trainer.save_model("medical_qa_model")
tokenizer.save_pretrained("medical_qa_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('medical_qa_model/tokenizer_config.json', 'medical_qa_model/tokenizer.json')

In [ ]:
# =========================================================
# 11. TẠO HỆ THỐNG QUESTION ANSWERING
# =========================================================

qa_pipeline = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer
)


# =========================================================
# 12. TEST MODEL
# =========================================================

context = """
Diabetes is a chronic disease that occurs when blood glucose is too high.
"""

question = "What is diabetes?"

result = qa_pipeline(
    question=question,
    context=context
)

print("Question:", question)
print("Answer:", result["answer"])
print("Confidence:", result["score"])




KeyError: "Unknown task question-answering, available tasks are ['any-to-any', 'audio-classification', 'automatic-speech-recognition', 'depth-estimation', 'document-question-answering', 'feature-extraction', 'fill-mask', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'keypoint-matching', 'mask-generation', 'ner', 'object-detection', 'sentiment-analysis', 'table-question-answering', 'text-classification', 'text-generation', 'text-to-audio', 'text-to-speech', 'token-classification', 'video-classification', 'zero-shot-audio-classification', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-object-detection']"

In [ ]:
!pip install -U transformers